In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [3]:
df = pd.read_csv("../data/raw/study_records.csv")

df.head()

,student_id,topic,exam_frequency,question_marks,difficulty,knowledge_before,study_minutes,quiz_before,quiz_after,exam_score
0,S001,TCP Congestion Control,9,10,8,25,90,30,82,86
1,S001,HTTP,8,8,6,40,60,45,88,91
2,S001,DNS,5,5,4,65,35,68,91,89
3,S001,Routing,8,8,7,30,60,35,78,84
4,S001,UDP,3,5,3,80,25,82,94,92


In [5]:
df["learning_gain"] = (
    df["quiz_after"] - df["quiz_before"]
)

In [6]:
df[["quiz_before", "quiz_after", "learning_gain"]].head()

,quiz_before,quiz_after,learning_gain
0,30,82,52
1,45,88,43
2,68,91,23
3,35,78,43
4,82,94,12


In [7]:
features = [
    "exam_frequency",
    "question_marks",
    "difficulty",
    "knowledge_before",
    "study_minutes",
    "quiz_before"
]

X = df[features]
y = df["learning_gain"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
model = LinearRegression()

model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](6,)","[ 1.96, 1.26,-3.64, 0.02, 0.14,-0.59]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](6,)","['exam_frequency','question_marks','difficulty','knowledge_before', 'study_minutes','quiz_before']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,55.13
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,6
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(6)


In [10]:
topic_df = df.groupby("topic").agg({
    "exam_frequency": "mean",
    "question_marks": "mean",
    "difficulty": "mean",
    "study_minutes": "mean"
}).reset_index()

topic_df

,topic,exam_frequency,question_marks,difficulty,study_minutes
0,DNS,5.0,5.0,4.0,33.0
1,HTTP,8.0,8.0,6.0,58.0
2,Routing,8.0,8.0,7.0,65.0
3,TCP Congestion Control,9.0,10.0,8.0,88.0
4,UDP,3.0,5.0,3.0,23.0


In [11]:
student_knowledge = {
    "TCP Congestion Control": 20,
    "HTTP": 35,
    "DNS": 70,
    "Routing": 30,
    "UDP": 85
}

In [12]:
topic_df["knowledge_before"] = topic_df["topic"].map(
    student_knowledge
)

In [13]:
topic_df

,topic,exam_frequency,question_marks,difficulty,study_minutes,knowledge_before
0,DNS,5.0,5.0,4.0,33.0,70
1,HTTP,8.0,8.0,6.0,58.0,35
2,Routing,8.0,8.0,7.0,65.0,30
3,TCP Congestion Control,9.0,10.0,8.0,88.0,20
4,UDP,3.0,5.0,3.0,23.0,85


In [14]:
topic_df["quiz_before"] = topic_df["knowledge_before"]

In [15]:
prediction_features = [
    "exam_frequency",
    "question_marks",
    "difficulty",
    "knowledge_before",
    "study_minutes",
    "quiz_before"
]

In [16]:
topic_df["predicted_gain"] = model.predict(
    topic_df[prediction_features]
)

In [17]:
topic_df["predicted_gain"] = model.predict(
    topic_df[prediction_features]
)

In [18]:
topic_df[[
    "topic",
    "study_minutes",
    "knowledge_before",
    "predicted_gain"
]]

,topic,study_minutes,knowledge_before,predicted_gain
0,DNS,33.0,70,21.711493
1,HTTP,58.0,35,47.389050
2,Routing,65.0,30,47.555226
3,TCP Congestion Control,88.0,20,57.272173
4,UDP,23.0,85,11.548657


In [19]:
topic_df["exam_value"] = (
    topic_df["exam_frequency"]
    * topic_df["question_marks"]
)

In [22]:
topic_df["expected_value"] = (
    topic_df["predicted_gain"]
    * topic_df["exam_value"]
)

In [23]:
topic_df["value_per_minute"] = (
    topic_df["expected_value"]
    / topic_df["study_minutes"]
)

In [24]:
ranked_topics = topic_df.sort_values(
    "value_per_minute",
    ascending=False
)

In [25]:
ranked_topics[[
    "topic",
    "predicted_gain",
    "study_minutes",
    "expected_value",
    "value_per_minute"
]]

,topic,predicted_gain,study_minutes,expected_value,value_per_minute
3,TCP Congestion Control,57.272173,88.0,5154.495569,58.573813
1,HTTP,47.389050,58.0,3032.899224,52.291366
2,Routing,47.555226,65.0,3043.534495,46.823608
0,DNS,21.711493,33.0,542.787335,16.448101
4,UDP,11.548657,23.0,173.229855,7.531733


In [26]:
def greedy_study_plan(topic_df, available_minutes):
    
    ranked = topic_df.sort_values(
        "value_per_minute",
        ascending=False
    )

    selected = []
    remaining = available_minutes

    for _, row in ranked.iterrows():

        if row["study_minutes"] <= remaining:

            selected.append(row)

            remaining -= row["study_minutes"]

    return pd.DataFrame(selected), remaining

In [27]:
plan, remaining = greedy_study_plan(
    topic_df,
    180
)

In [29]:
plan[[
    "topic",
    "study_minutes",
    "predicted_gain",
    "value_per_minute"
]]

,topic,study_minutes,predicted_gain,value_per_minute
3,TCP Congestion Control,88.0,57.272173,58.573813
1,HTTP,58.0,47.389050,52.291366
0,DNS,33.0,21.711493,16.448101


In [30]:
print("Remaining minutes:", remaining)

Remaining minutes: 1.0


In [31]:
def show_plan(plan, available_minutes):

    print("=" * 50)
    print("       EMERGENCY STUDY PLAN")
    print("=" * 50)

    print(f"Available time: {available_minutes} minutes")
    print()

    for i, (_, row) in enumerate(plan.iterrows(), start=1):

        print(f"{i}. {row['topic']}")
        print(f"   Study time: {row['study_minutes']:.0f} min")
        print(f"   Predicted gain: {row['predicted_gain']:.1f}")
        print(f"   Value/minute: {row['value_per_minute']:.2f}")
        print()

    used = plan["study_minutes"].sum()

    print("-" * 50)
    print(f"Used time: {used:.0f} minutes")
    print(f"Remaining: {available_minutes - used:.0f} minutes")

In [32]:
show_plan(plan, 180)

       EMERGENCY STUDY PLAN
Available time: 180 minutes

1. TCP Congestion Control
   Study time: 88 min
   Predicted gain: 57.3
   Value/minute: 58.57

2. HTTP
   Study time: 58 min
   Predicted gain: 47.4
   Value/minute: 52.29

3. DNS
   Study time: 33 min
   Predicted gain: 21.7
   Value/minute: 16.45

--------------------------------------------------
Used time: 179 minutes
Remaining: 1 minutes


In [33]:
plan, remaining = greedy_study_plan(
    topic_df,
    60
)

show_plan(plan, 60)

       EMERGENCY STUDY PLAN
Available time: 60 minutes

1. HTTP
   Study time: 58 min
   Predicted gain: 47.4
   Value/minute: 52.29

--------------------------------------------------
Used time: 58 minutes
Remaining: 2 minutes


In [34]:
plan, remaining = greedy_study_plan(
    topic_df,
    120
)

show_plan(plan, 120)

       EMERGENCY STUDY PLAN
Available time: 120 minutes

1. TCP Congestion Control
   Study time: 88 min
   Predicted gain: 57.3
   Value/minute: 58.57

2. UDP
   Study time: 23 min
   Predicted gain: 11.5
   Value/minute: 7.53

--------------------------------------------------
Used time: 111 minutes
Remaining: 9 minutes


In [36]:
plan, remaining = greedy_study_plan(
    topic_df,
    240
)

show_plan(plan, 240)

       EMERGENCY STUDY PLAN
Available time: 240 minutes

1. TCP Congestion Control
   Study time: 88 min
   Predicted gain: 57.3
   Value/minute: 58.57

2. HTTP
   Study time: 58 min
   Predicted gain: 47.4
   Value/minute: 52.29

3. Routing
   Study time: 65 min
   Predicted gain: 47.6
   Value/minute: 46.82

4. UDP
   Study time: 23 min
   Predicted gain: 11.5
   Value/minute: 7.53

--------------------------------------------------
Used time: 234 minutes
Remaining: 6 minutes


In [41]:
def optimize_study_plan(topic_df, available_minutes):

    topics = topic_df.reset_index(drop=True)

    n = len(topics)
    capacity = int(available_minutes)

    dp = np.zeros((n + 1, capacity + 1))

    for i in range(1, n + 1):

        time = int(topics.loc[i - 1, "study_minutes"])
        value = topics.loc[i - 1, "expected_value"]

        for t in range(capacity + 1):

            dp[i][t] = dp[i - 1][t]

            if time <= t:

                dp[i][t] = max(
                    dp[i][t],
                    dp[i - 1][t - time] + value
                )

    selected_indices = []

    t = capacity

    for i in range(n, 0, -1):

        if dp[i][t] != dp[i - 1][t]:

            selected_indices.append(i - 1)

            t -= int(
                topics.loc[i - 1, "study_minutes"]
            )

    selected_indices.reverse()

    selected_topics = topics.loc[
        selected_indices
    ]

    return selected_topics, dp[n][capacity]

In [42]:
optimized_plan, total_value = optimize_study_plan(
    topic_df,
    180
)

In [43]:
optimized_plan[[
    "topic",
    "study_minutes",
    "predicted_gain",
    "expected_value"
]]

,topic,study_minutes,predicted_gain,expected_value
0,DNS,33.0,21.711493,542.787335
1,HTTP,58.0,47.389050,3032.899224
3,TCP Congestion Control,88.0,57.272173,5154.495569


In [44]:
print("Total expected value:", total_value)

Total expected value: 8730.182127653918


In [45]:
days_remaining = 1

In [46]:
available_minutes = 180

In [47]:
print(f"Days remaining: {days_remaining}")
print(f"Available study time: {available_minutes} minutes")

Days remaining: 1
Available study time: 180 minutes


In [48]:
def priority_label(value):

    if value >= 100:
        return "VERY HIGH"

    elif value >= 50:
        return "HIGH"

    elif value >= 20:
        return "MEDIUM"

    else:
        return "LOW"

In [49]:
optimized_plan["priority"] = (
    optimized_plan["expected_value"]
    .apply(priority_label)
)

In [50]:
optimized_plan[[
    "topic",
    "study_minutes",
    "predicted_gain",
    "priority"
]]

,topic,study_minutes,predicted_gain,priority
0,DNS,33.0,21.711493,VERY HIGH
1,HTTP,58.0,47.389050,VERY HIGH
3,TCP Congestion Control,88.0,57.272173,VERY HIGH


In [51]:
def generate_reason(row):

    reasons = []

    if row["exam_frequency"] >= 8:
        reasons.append("frequently appears in exams")

    if row["knowledge_before"] <= 40:
        reasons.append("your current knowledge is low")

    if row["question_marks"] >= 8:
        reasons.append("has high mark potential")

    if row["difficulty"] >= 7:
        reasons.append("is relatively difficult")

    if len(reasons) == 0:
        return "Moderate priority based on available data."

    return " + ".join(reasons)

In [52]:
optimized_plan["reason"] = optimized_plan.apply(
    generate_reason,
    axis=1
)

In [53]:
optimized_plan[[
    "topic",
    "priority",
    "reason"
]]

,topic,priority,reason
0,DNS,VERY HIGH,Moderate priority based on available data.
1,HTTP,VERY HIGH,frequently appears in exams + your current kno...
3,TCP Congestion Control,VERY HIGH,frequently appears in exams + your current kno...


In [54]:
student_knowledge_2 = {
    "TCP Congestion Control": 80,
    "HTTP": 20,
    "DNS": 30,
    "Routing": 75,
    "UDP": 90
}

In [55]:
topic_df["knowledge_before"] = topic_df["topic"].map(
    student_knowledge_2
)

In [56]:
topic_df["quiz_before"] = topic_df["knowledge_before"]

In [57]:
prediction_features = [
    "exam_frequency",
    "question_marks",
    "difficulty",
    "knowledge_before",
    "study_minutes",
    "quiz_before"
]

topic_df["predicted_gain"] = model.predict(
    topic_df[prediction_features]
)

In [58]:
topic_df["exam_value"] = (
    topic_df["exam_frequency"]
    * topic_df["question_marks"]
)

topic_df["expected_value"] = (
    topic_df["predicted_gain"]
    * topic_df["exam_value"]
)

topic_df["value_per_minute"] = (
    topic_df["expected_value"]
    / topic_df["study_minutes"]
)

In [59]:
optimized_plan_2, total_value_2 = optimize_study_plan(
    topic_df,
    180
)

In [60]:
time_results = []

for minutes in [30, 60, 120, 180, 300]:

    plan, total_value = optimize_study_plan(
        topic_df,
        minutes
    )

    time_results.append({
        "Available Time": minutes,
        "Selected Topics": ", ".join(plan["topic"].tolist()),
        "Total Study Time": plan["study_minutes"].sum(),
        "Expected Value": total_value
    })

time_results_df = pd.DataFrame(time_results)

time_results_df

,Available Time,Selected Topics,Total Study Time,Expected Value
0,30,UDP,23.0,130.817336
1,60,HTTP,58.0,3575.779471
2,120,"DNS, HTTP, UDP",114.0,4814.884400
3,180,"DNS, HTTP, TCP Congestion Control",179.0,6784.861241
4,300,"DNS, HTTP, Routing, TCP Congestion Control, UDP",267.0,8330.572329


In [61]:
comparison_results = []

for minutes in [60, 120, 180, 240]:

    greedy_plan, greedy_remaining = greedy_study_plan(
        topic_df,
        minutes
    )

    greedy_value = greedy_plan["expected_value"].sum()

    knapsack_plan, knapsack_value = optimize_study_plan(
        topic_df,
        minutes
    )

    comparison_results.append({
        "Available Time": minutes,
        "Greedy Value": greedy_value,
        "Knapsack Value": knapsack_value,
        "Greedy Topics": ", ".join(greedy_plan["topic"].tolist()),
        "Knapsack Topics": ", ".join(knapsack_plan["topic"].tolist())
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,Available Time,Greedy Value,Knapsack Value,Greedy Topics,Knapsack Topics
0,60,3575.779471,3575.779471,HTTP,HTTP
1,120,4814.884400,4814.884400,"HTTP, DNS, UDP","DNS, HTTP, UDP"
2,180,6784.861241,6784.861241,"HTTP, DNS, TCP Congestion Control","DNS, HTTP, TCP Congestion Control"
3,240,6915.678576,7222.284736,"HTTP, DNS, TCP Congestion Control, UDP","HTTP, Routing, TCP Congestion Control, UDP"


In [62]:
student_knowledge = {
    "TCP Congestion Control": 0,
    "HTTP": 0,
    "DNS": 0,
    "Routing": 0,
    "UDP": 0
}

In [63]:
student_knowledge = {
    "TCP Congestion Control": 100,
    "HTTP": 100,
    "DNS": 100,
    "Routing": 100,
    "UDP": 100
}

In [64]:
available_minutes = 10

In [65]:
available_minutes = 600

In [66]:
days_remaining = 1

In [67]:
available_minutes = 180

## Day 4 — My Observations

### What I learned

1. I learned how to combine ML predictions with exam importance.
2. I learned how to rank topics using value per minute.
3. I learned how to use Greedy and Knapsack optimization for study planning.

### Greedy vs Knapsack

The Greedy planner is simple and fast, while the Knapsack optimizer tries to find the best combination of topics within the available study time.

### What surprised me

Changing the student's current knowledge and available study time changes the recommended topics.

### Current limitations

The model predicts learning gain rather than direct exam-score improvement. The current dataset is also small and the exam value calculation is heuristic.

### My idea for improving the system

In the future, I want to use real diagnostic quiz results and predict expected exam-score improvement for each topic and study duration.